# ReelForge Wan2GP worker — one claimed scene

Same checkout, dependencies, and `wgp.py` command as the verified one-shot notebook. After the model-load gate, this notebook claims one `provider=wan` job, writes that job's prompt into the settings file, generates one 81-frame clip, and uploads it. A failure does not change resolution, frames, steps, seed, attention, or profile.


Clean notebook for one Tesla T4 generation. It checks out the official repository and stops when a documented command would replace the running PyTorch build, when a checkpoint size cannot be measured, or when the requested settings are not present in the source.

This file does not claim that a video has been produced. The FFprobe cell prints a pass only after an MP4 from this session has a video stream.

Official repository: `https://github.com/deepbeepmeep/Wan2GP.git`  
Branch: `main`  
Commit: `b8b18f8114e432eea8f3d7e853a51dd91fa99571`

Requested generation settings, checked against that commit before they are written:

- Model: Wan 2.1 text-to-video 1.3B (`t2v_1.3B`)
- Resolution: 416×240
- Frames: 81, which is 5.0625 seconds at the model's 16 fps
- Output copy: `/kaggle/working/result_camera_motion.mp4`. `result.mp4` and `result_5sec.mp4` are not written.
- Steps: 20
- Precision: FP16 via `--fp16`
- Attention: SDPA via `--attention sdpa`
- GPU: the first T4, `cuda:0`, with `CUDA_VISIBLE_DEVICES=0`
- Profile: 5, only after the source still lists that profile and memory is at least its stated floor


## 1. Environment and GPU detection

Read the Kaggle machine. Do not install packages and do not download weights.


In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys

ROOT = Path("/kaggle/working")
if not ROOT.is_dir():
    raise SystemExit("This notebook is for a Kaggle session. /kaggle/working is not present, so nothing was cloned or installed.")

PREFLIGHT = ROOT / "wan2gp_preflight"
PREFLIGHT.mkdir(exist_ok=True)
for stale in PREFLIGHT.glob("*.ok"):
    stale.unlink()

print("python", sys.version.replace("\n", " "))
print("executable", sys.executable)
print("platform", platform.platform())

def run(cmd):
    completed = subprocess.run(cmd, check=False, text=True, capture_output=True)
    print("$", " ".join(cmd))
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise SystemExit(f"Command failed ({completed.returncode}): {' '.join(cmd)}")
    return completed.stdout

run(["nvidia-smi"])
names = run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"]).strip().splitlines()
if not names or not names[0].strip():
    raise SystemExit("nvidia-smi did not report a GPU.")
first = names[0]
if "T4" not in first:
    raise SystemExit(f"The first GPU is not a Tesla T4: {first}")
print(f"GPU count reported: {len(names)}. Generation will use index 0 only.")

for binary in ("ffmpeg", "ffprobe"):
    path = shutil.which(binary)
    if not path:
        raise SystemExit(f"{binary} is not on PATH.")
    print(binary, path)
    run([binary, "-version"])

usage = shutil.disk_usage(ROOT)
meminfo = Path("/proc/meminfo").read_text(encoding="utf-8", errors="replace")
mem_total = next(line for line in meminfo.splitlines() if line.startswith("MemTotal:"))
print(f"disk_total_bytes={usage.total} disk_used_bytes={usage.used} disk_free_bytes={usage.free}")
print(mem_total)
(PREFLIGHT / "environment.ok").write_text("ok\n", encoding="utf-8")
print("environment_check=passed")


## 2. Repository clone and exact version checkout

Clone the official repository into `/kaggle/working/Wan2GP` and detach that exact commit. The committed OmniVoice file begins with a UTF-8 BOM. Python 3.12 raises `SyntaxError` on U+FEFF, and `wgp.py` imports that module while registering model families. The cell removes those three bytes only when the following bytes are the file's shebang, then compiles the remainder. No other upstream file is edited.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path("/kaggle/working")
PREFLIGHT = ROOT / "wan2gp_preflight"
if not (PREFLIGHT / "environment.ok").is_file():
    raise SystemExit("Environment check has not passed.")

COMMIT = "b8b18f8114e432eea8f3d7e853a51dd91fa99571"
REPO = ROOT / "Wan2GP"
REMOTE = "https://github.com/deepbeepmeep/Wan2GP.git"

def git(*args, check=True):
    cmd = ["git", "-C", str(REPO), *args] if args and args[0] != "clone" else ["git", *args]
    if args and args[0] == "clone":
        cmd = ["git", *args]
    print("$", " ".join(cmd))
    completed = subprocess.run(cmd, check=False, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise SystemExit(f"git failed ({completed.returncode}): {' '.join(cmd)}")
    return completed.stdout.strip()

if not (REPO / ".git").is_dir():
    print("$", "git", "clone", "--depth", "1", REMOTE, str(REPO))
    completed = subprocess.run(["git", "clone", "--depth", "1", REMOTE, str(REPO)], check=False, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise SystemExit("git clone failed.")

head = git("rev-parse", "HEAD")
if head != COMMIT:
    git("fetch", "--depth", "1", "origin", COMMIT)
    git("checkout", "--detach", COMMIT)
head = git("rev-parse", "HEAD")
if head != COMMIT:
    raise SystemExit(f"HEAD is {head}, not {COMMIT}.")
print("git_head", head)

omnivoice = REPO / "models" / "TTS" / "omnivoice" / "modeling_omnivoice.py"
data = omnivoice.read_bytes()
if data.startswith(b"\xef\xbb\xbf"):
    rest = data[3:]
    if not rest.startswith(b"#!/usr/bin/env python3"):
        raise SystemExit("OmniVoice starts with a UTF-8 BOM, but the following bytes are not the expected shebang. The file was not modified.")
    omnivoice.write_bytes(rest)
    compile(rest.decode("utf-8"), str(omnivoice), "exec")
    print("omnivoice_bom=removed_3_bytes")
elif data.startswith(b"#!"):
    compile(data.decode("utf-8"), str(omnivoice), "exec")
    print("omnivoice_bom=absent")
else:
    raise SystemExit("Unexpected OmniVoice header. The file was not modified.")

changed = git("diff", "--name-only")
allowed = {"models/TTS/omnivoice/modeling_omnivoice.py"}
names = {line.strip() for line in changed.splitlines() if line.strip()}
if not names <= allowed:
    raise SystemExit(f"Unexpected source changes after checkout: {sorted(names)}")
print("source_diff", sorted(names) or "clean")
(PREFLIGHT / "checkout.ok").write_text(head + "\n", encoding="utf-8")
print("checkout_check=passed")


## 3. Python, CUDA, PyTorch, and GPU compatibility

`docs/INSTALLATION.md` recommends Python 3.11.14 and `torch==2.10.0` from the cu130 index for RTX 20-series and newer, and Python 3.10.9 with PyTorch 2.7.1 cu128 for GTX 10-series. This cell does not run either install. A Tesla T4 is Turing (`sm_75`). `wgp.py` forces the FP16 path when the compute capability major version is below 8. `--fp16`, `--attention`, `--profile`, `--gpu`, and `--process` are read from `shared/cli_args.py`.

Profile 5 is used only when `wgp.py` still contains that choice and both its stated floors are met: 24 GB RAM and 10 GB VRAM. The requested 416×240 frame size is accepted only while the non-5B Wan VAE block size remains 16 and both sides divide by 16. Eighty-one frames is accepted only while the non-VACE minimum is 5, the step is 4, and the frame offset default is 1, which leaves `floor_frame_count(81, 5, 4, 1)` at 81. At the non-5B fps of 16 that is 5.0625 seconds. `t2v_1.3B` does not use the recam length lock.

`docs/CLI.md` mentions `--quantize-transformer`. `shared/cli_args.py` does not define it. This notebook does not pass that flag.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path("/kaggle/working")
PREFLIGHT = ROOT / "wan2gp_preflight"
REPO = ROOT / "Wan2GP"
if (PREFLIGHT / "checkout.ok").read_text(encoding="utf-8").strip() != "b8b18f8114e432eea8f3d7e853a51dd91fa99571":
    raise SystemExit("Checkout check has not passed.")

def need(path):
    file_path = REPO / path
    if not file_path.is_file():
        raise SystemExit(f"Missing repository file: {path}")
    return file_path.read_text(encoding="utf-8", errors="replace")

installation = need("docs/INSTALLATION.md")
getting_started = need("docs/GETTING_STARTED.md")
cli_args = need("shared/cli_args.py")
wgp = need("wgp.py")
handler = need("models/wan/wan_handler.py")
scheduler = need("shared/utils/frame_scheduler.py")
defaults = need("defaults/t2v_1.3B.json")

for phrase in (
    "Python 3.11.14",
    "torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu130",
    "Wan 2.1 T2V 1.3B",
    "lowest VRAM (6GB)",
):
    blob = installation + "\n" + getting_started
    if phrase not in blob:
        raise SystemExit(f"Official documentation no longer contains the expected phrase: {phrase}")

print("DOCUMENTED_STACK Python 3.11.14 + torch 2.10.0 cu130 for RTX 20-series and newer.")
print("RUNNING_PYTHON", sys.version.split()[0])
if sys.version_info < (3, 10):
    raise SystemExit("Python is older than 3.10. Stopping.")
if sys.version_info[:2] != (3, 11):
    print("DEVIATION Python is not the documented 3.11.14. Kaggle's interpreter is left unchanged.")

import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), "compiled", torch.version.cuda)
if not torch.cuda.is_available():
    raise SystemExit("CUDA is not available. Stopping before any install or download.")
if not str(torch.__version__).startswith("2.10.0"):
    raise SystemExit(f"PyTorch is {torch.__version__}, not the documented 2.10.0 line. Stopping rather than reinstalling it.")
if "+cu128" not in str(torch.__version__):
    print("DEVIATION torch is 2.10.0 but not the cu128 build already on this image:", torch.__version__)
print("DEVIATION The documented cu130 wheel will not be installed. It would replace the current CUDA build.")
major, minor = torch.cuda.get_device_capability(0)
print(f"compute_capability={major}.{minor} name={torch.cuda.get_device_name(0)}")
if major >= 8:
    raise SystemExit("This GPU reports compute capability 8 or higher. The FP16 switch in wgp.py would not apply. Stopping instead of changing the precision request.")
if "Switching to FP16 models when possible" not in wgp:
    raise SystemExit("wgp.py no longer contains the FP16 switch for GPUs below compute capability 8.")

for flag in ("--fp16", "--attention", "--profile", "--gpu", "--process", "--output-dir", "--verbose", "--dry-run"):
    if flag not in cli_args:
        raise SystemExit(f"shared/cli_args.py does not define {flag}.")
if "--quantize-transformer" in cli_args:
    raise SystemExit("shared/cli_args.py now defines --quantize-transformer. Re-read it before passing a quantization flag.")
print("quantize_transformer_flag=absent_from_parser")

profile_line = 'Profile 5, VerylowRAM_LowVRAM (Fail safe): at least 24 GB of RAM and 10 GB of VRAM'
if profile_line not in wgp:
    raise SystemExit("Profile 5 is not the documented fail-safe in wgp.py. The requested profile was not changed.")
meminfo = Path("/proc/meminfo").read_text(encoding="utf-8", errors="replace")
mem_kb = int(next(line for line in meminfo.splitlines() if line.startswith("MemTotal:")).split()[1])
vram = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits", "-i", "0"],
    check=True, text=True, capture_output=True,
)
vram_mib = int(vram.stdout.strip().splitlines()[0])
print(f"mem_total_kib={mem_kb} gpu0_vram_mib={vram_mib}")
if mem_kb < 24 * 1024 * 1024 or vram_mib < 10 * 1024:
    raise SystemExit("Memory is below Profile 5's stated floor of 24 GB RAM and 10 GB VRAM. Stopping.")
print("profile=5 matches the source choice and its stated memory floor.")
if "at least 32 GB of RAM and 12 GB of VRAM" not in wgp:
    raise SystemExit("Profile 4's memory text changed. Re-read the profile list before keeping profile 5.")

if 'frames_minimum, frames_steps = 5, 4' not in handler:
    raise SystemExit("Wan non-VACE frame minimum/step is no longer 5 and 4. 81 frames was not rewritten.")
if 'model_def.get("frames_offset", 1)' not in wgp:
    raise SystemExit("The frame offset default is no longer 1.")
if "def floor_frame_count" not in scheduler:
    raise SystemExit("floor_frame_count is missing.")
if 'extra_model_def["vae_block_size"] = 32 if test_wan_5B(base_model_type) or base_model_type in ["scail"] or scail2 else 16' not in handler:
    raise SystemExit("The non-5B VAE block size is no longer 16. 416x240 was not rewritten.")
width, height = 416, 240
if width % 16 or height % 16:
    raise SystemExit("416x240 is not divisible by 16.")

def floor_frame_count(frame_count, minimum, step, offset=1):
    frame_count = max(minimum, frame_count)
    step = max(1, step)
    offset = max(0, offset)
    if step <= 1:
        return frame_count
    lower = ((frame_count - offset) // step) * step + offset
    return lower if lower >= minimum else minimum

if floor_frame_count(17, 5, 4, 1) != 17:
    raise SystemExit("17 is not on the 5/4/1 frame grid. The frame count was not rewritten.")
if floor_frame_count(81, 5, 4, 1) != 81:
    raise SystemExit("81 is not on the 5/4/1 frame grid. The frame count was not rewritten.")
print("frame_count=81 duration_seconds=5.0625 fps=16")
if '"architecture": "t2v_1.3B"' not in defaults or "wan2.1_text2video_1.3B_mbf16.safetensors" not in defaults:
    raise SystemExit("defaults/t2v_1.3B.json does not point at the 1.3B text-to-video checkpoint.")
if '"text_encoder_quantization" : "int8"' not in wgp:
    raise SystemExit("The default text-encoder quantization is no longer int8. Stopping before selecting an encoder file.")
if "models_t5_umt5-xxl-enc-quanto_int8.safetensors" not in handler or "models_t5_umt5-xxl-enc-bf16.safetensors" not in handler:
    raise SystemExit("The Wan handler no longer lists both UMT5 encoder files.")
print("settings_check=passed")
print("text_encoder=quanto_int8_by_default_config")
print("bf16_text_encoder=not_selected")
(PREFLIGHT / "compatibility.ok").write_text("ok\n", encoding="utf-8")


## 4. Disk space and memory preflight

Print free disk and RAM before dependencies and again before checkpoint bytes are accepted. This cell does not download model files.


In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/kaggle/working")
PREFLIGHT = ROOT / "wan2gp_preflight"
if not (PREFLIGHT / "compatibility.ok").is_file():
    raise SystemExit("Compatibility check has not passed.")
usage = shutil.disk_usage(ROOT)
meminfo = Path("/proc/meminfo").read_text(encoding="utf-8", errors="replace")
print(f"disk_total_bytes={usage.total} disk_used_bytes={usage.used} disk_free_bytes={usage.free}")
for key in ("MemTotal:", "MemAvailable:"):
    print(next(line for line in meminfo.splitlines() if line.startswith(key)))
(PREFLIGHT / "disk_snapshot.ok").write_text(str(usage.free) + "\n", encoding="utf-8")
print("disk_snapshot=recorded")


## 5. Minimal dependency installation

Versions below are copied from `requirements.txt` at this commit and limited to packages imported while `wgp.py` starts or while the Wan 2.1 text-to-video modules load.

Not installed:

- The cu130 PyTorch command from `docs/INSTALLATION.md`.
- `pip install -r requirements.txt`. That file enables a CUDA 13 ONNX nightly index and pins `onnxruntime-gpu==1.25.0.dev20260210001`, `numpy==2.1.2`, `rembg[gpu]`, and packages for other model families.
- `numpy==2.1.2`. The image copy stays in place. Upgrading it is a separate decision.
- `onnxruntime-gpu==1.25.0.dev20260210001` and the `rembg[gpu]` extra. `rembg` imports `onnxruntime` at startup. The notebook installs the CPU package `onnxruntime==1.30.0`, already run on this Kaggle image without changing torch. The CUDA 13 nightly GPU build stays refused.
- SageAttention and FlashAttention. SDPA is the requested backend, and `shared/attention.py` always registers it.

`openai-whisper==20250625` is installed because the committed `shared/deepy/transcription.py` executes `import whisper` while `wgp.py` loads. The transcription file is not patched.

The dry-run must not propose an install or upgrade of `torch`, `torchvision`, `torchaudio`, `numpy`, `triton`, `onnxruntime-gpu`, or an NVIDIA CUDA package. If it does, the cell stops and installs nothing.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path("/kaggle/working")
PREFLIGHT = ROOT / "wan2gp_preflight"
if not (PREFLIGHT / "disk_snapshot.ok").is_file():
    raise SystemExit("Disk snapshot has not been recorded.")

packages = [
    "mmgp==3.8.2",
    "diffusers==0.36.0",
    "transformers==4.54.0",
    "tokenizers>=0.20.3",
    "accelerate>=1.1.1",
    "gradio==5.29.0",
    "opencv-python>=4.12.0.88",
    "einops",
    "ftfy",
    "easydict",
    "sentencepiece",
    "omegaconf",
    "imageio",
    "imageio-ffmpeg",
    "ffmpeg-python",
    "soundfile",
    "tqdm",
    "tiktoken",
    "openai-whisper==20250625",
    "rembg==2.0.65",
    "onnxruntime==1.30.0",
    "chumpy @ https://github.com/deepbeepmeep/chumpy/releases/download/v0.71/chumpy-0.71-py3-none-any.whl",
    "smplfitter @ https://github.com/deepbeepmeep/smplfitter/releases/download/v0.2.10/smplfitter-0.2.10-py3-none-any.whl",
    "loguru",
    "av",
    "matplotlib",
    "pydantic==2.11.10",
    "gitpython==3.1.45",
    "huggingface_hub[hf_xet]",
    "hf_xet>=1.5.2",
]
report_path = PREFLIGHT / "pip-dry-run.json"
cmd = [
    sys.executable, "-m", "pip", "install",
    "--dry-run", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "--report", str(report_path),
    *packages,
]
print("$", " ".join(cmd))
completed = subprocess.run(cmd, check=False, text=True)
if completed.returncode != 0 or not report_path.is_file():
    raise SystemExit("pip dry-run did not produce a report. Nothing was installed.")
report = json.loads(report_path.read_text(encoding="utf-8"))
forbidden = {"torch", "torchvision", "torchaudio", "numpy", "triton", "onnxruntime-gpu"}
blocked = []
for item in report.get("install", []):
    name = item.get("metadata", {}).get("name", "").lower().replace("_", "-")
    print("dry_run_would_install", name)
    if name in forbidden or name.startswith("nvidia-") or name.startswith("cuda"):
        blocked.append(name)
if blocked:
    raise SystemExit("Dry-run would change protected packages: " + ", ".join(blocked) + ". Nothing was installed.")

install = [arg for arg in cmd if arg != "--dry-run" and not arg.endswith("pip-dry-run.json") and arg != "--report"]
print("$", " ".join(install))
installed = subprocess.run(install, check=False, text=True)
if installed.returncode != 0:
    raise SystemExit("pip install failed. Checkpoint download will not start.")
(PREFLIGHT / "dependencies.ok").write_text("ok\n", encoding="utf-8")
print("dependency_install=passed")


## 6. Dependency and import validation

Write one settings object from `models/_settings.json`, overriding only the checked fields. Then run `wgp.py --process ... --dry-run`. That loads the application and validates the task. `--dry-run` exits before `process_tasks_cli`, so it does not download the checkpoints or generate a video. A failed import stops the notebook here.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = Path("/kaggle/working")
PREFLIGHT = ROOT / "wan2gp_preflight"
REPO = ROOT / "Wan2GP"
if not (PREFLIGHT / "dependencies.ok").is_file():
    raise SystemExit("Dependency install has not passed.")

required = [
    "torch", "numpy", "gradio", "mmgp", "cv2", "PIL", "einops", "ftfy", "easydict",
    "regex", "sentencepiece", "omegaconf", "diffusers", "transformers", "whisper",
    "rembg", "imageio", "ffmpeg", "soundfile", "accelerate", "safetensors",
]
probe = subprocess.run([sys.executable, "-c", "import " + ", ".join(required)], check=False, text=True, capture_output=True)
print(probe.stdout)
print(probe.stderr)
if probe.returncode != 0:
    raise SystemExit(
        "A required import failed. Stopping before checkpoint download."
    )

settings_template = json.loads((REPO / "models" / "_settings.json").read_text(encoding="utf-8"))
settings_template.update({
    "model_type": "t2v_1.3B",
    "prompt": (
        "A photorealistic cinematic architectural film of an ultra-luxurious modern oceanfront living room. "
        "The camera performs a clearly visible, smooth and continuous forward dolly movement, entering the living room "
        "from a wide entrance and gradually approaching the floor-to-ceiling glass windows. During the movement, the "
        "camera gently pans to the right, revealing the complete elegant interior, a premium beige sectional sofa, "
        "sculptural furniture, polished marble flooring and sophisticated gold details. Sheer white curtains visibly "
        "sway in the ocean breeze. Outside, palm tree leaves move naturally in the wind and ocean waves visibly roll "
        "toward the shore. Sunlight creates subtle moving reflections across the marble floor. Maintain realistic "
        "perspective, stable architecture, consistent furniture placement and natural motion throughout the entire shot. "
        "High-end luxury real estate commercial, cinematic camera operation, photorealistic lighting, realistic textures, "
        "smooth motion, professional architectural cinematography. No people, no cuts, no scene transitions, no sudden "
        "camera movement, no flickering, no warped walls, no distorted windows, no duplicated furniture."
    ),
    "resolution": "416x240",
    "video_length": 81,
    "num_inference_steps": 20,
    "seed": 1,
    "repeat_generation": 1,
    "prompt_enhancer": "",
    "spatial_upsampling": "",
    "temporal_upsampling": "",
    "override_profile": 5,
    "override_attention": "sdpa",
    "image_mode": 0,
})
settings_path = ROOT / "t2v_1_3B_settings.json"
settings_path.write_text(json.dumps(settings_template, indent=2) + "\n", encoding="utf-8")
print("settings_json", settings_path)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["TOKENIZERS_PARALLELISM"] = "false"
cmd = [
    sys.executable, "wgp.py",
    "--process", str(settings_path),
    "--dry-run",
    "--gpu", "cuda:0",
    "--attention", "sdpa",
    "--profile", "5",
    "--fp16",
    "--verbose", "2",
]
print("$", "CUDA_VISIBLE_DEVICES=0", " ".join(cmd))
completed = subprocess.run(cmd, cwd=REPO, env=env, check=False, text=True)
if completed.returncode != 0:
    raise SystemExit("wgp.py --dry-run failed. Checkpoint download will not start.")
(PREFLIGHT / "import.ok").write_text("ok\n", encoding="utf-8")
print("import_validation=passed")


## 7. Model checkpoint selection and download-size preflight

The transformer URL comes from `defaults/t2v_1.3B.json`. The text encoder is the int8 UMT5 file named in `models/wan/wan_handler.py`, because a new `wgp_config.json` sets `text_encoder_quantization` to `int8` and `get_model_filename` prefers that token. The BF16 UMT5 file is listed in the handler and is not part of this selection.

`query_model_files` also names the UMT5 tokenizer, the CLIP bundle, `Wan2.1_VAE.safetensors`, and `Wan2.1_VAE_upscale2x_imageonly_real_v1.safetensors` for every non-5B Wan model. The application downloads those files during model load. This notebook does not remove them. The upscaler and CLIP weights are not used by this text-to-video pass while spatial upsampling is empty, but they are still required by the unmodified loader.

Sizes come from HTTP `Content-Length`. A `.safetensors` file without that header stops the notebook. The run stops unless free space on `/kaggle/working` exceeds the measured total plus 1 GiB. No response body is downloaded here.


In [ ]:
from pathlib import Path
import shutil
import urllib.request

ROOT = Path("/kaggle/working")
PREFLIGHT = ROOT / "wan2gp_preflight"
REPO = ROOT / "Wan2GP"
if not (PREFLIGHT / "import.ok").is_file():
    raise SystemExit("Import validation has not passed.")

handler = (REPO / "models" / "wan" / "wan_handler.py").read_text(encoding="utf-8")
defaults = (REPO / "defaults" / "t2v_1.3B.json").read_text(encoding="utf-8")
bf16_name = "models_t5_umt5-xxl-enc-bf16.safetensors"
int8_name = "models_t5_umt5-xxl-enc-quanto_int8.safetensors"
transformer_name = "wan2.1_text2video_1.3B_mbf16.safetensors"
if bf16_name not in handler or int8_name not in handler or transformer_name not in defaults:
    raise SystemExit("Checkpoint names in the checkout do not match the selection rule. Nothing was downloaded.")

repo_files = {
    "DeepBeepMeep/Wan2.1": [
        transformer_name,
        f"umt5-xxl/{int8_name}",
        "umt5-xxl/special_tokens_map.json",
        "umt5-xxl/spiece.model",
        "umt5-xxl/tokenizer.json",
        "umt5-xxl/tokenizer_config.json",
        "xlm-roberta-large/models_clip_open-clip-xlm-roberta-large-vit-huge-14-bf16.safetensors",
        "xlm-roberta-large/sentencepiece.bpe.model",
        "xlm-roberta-large/special_tokens_map.json",
        "xlm-roberta-large/tokenizer.json",
        "xlm-roberta-large/tokenizer_config.json",
        "Wan2.1_VAE.safetensors",
        "Wan2.1_VAE_upscale2x_imageonly_real_v1.safetensors",
    ]
}
for relative in repo_files["DeepBeepMeep/Wan2.1"]:
    name = relative.split("/")[-1]
    if name == bf16_name or (name not in handler and name not in defaults):
        raise SystemExit(f"Refusing an unlisted checkpoint name: {relative}")

measured = []
unknown_small = []
for repo_id, relatives in repo_files.items():
    for relative in relatives:
        url = f"https://huggingface.co/{repo_id}/resolve/main/{relative}"
        request = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(request, timeout=60) as response:
            length = response.headers.get("Content-Length")
        print(f"head {relative} content_length={length}")
        if length is None or not length.isdigit() or int(length) <= 0:
            if relative.endswith(".safetensors"):
                raise SystemExit(f"No Content-Length for {relative}. Stopping before download.")
            unknown_small.append(relative)
            continue
        measured.append((relative, int(length)))

total = sum(size for _, size in measured)
margin = 1024 ** 3
usage = shutil.disk_usage(ROOT)
print(f"selected_known_bytes={total}")
print(f"unknown_non_safetensors={unknown_small}")
print(f"disk_free_bytes={usage.free} required_free_bytes={total + margin}")
if usage.free <= total + margin:
    raise SystemExit("Not enough free space for the selected checkpoints plus a 1 GiB margin. Nothing was downloaded.")
(PREFLIGHT / "checkpoints.ok").write_text(str(total) + "\n", encoding="utf-8")
print("checkpoint_preflight=passed")


## 8. Model loading

`wgp.py --process` downloads the selected files into `ckpts/` and loads them in the same process that generates. `files_locator.get_download_location` uses that `ckpts` directory. A separate `--test` launch would load the model and then the generation command would load it again. This cell only confirms the size gate and points the Hugging Face cache at `Wan2GP/ckpts` so the cache is on the same volume as the loader. It does not load weights.


In [ ]:
from pathlib import Path
import os

ROOT = Path("/kaggle/working")
PREFLIGHT = ROOT / "wan2gp_preflight"
REPO = ROOT / "Wan2GP"
if not (PREFLIGHT / "checkpoints.ok").is_file():
    raise SystemExit("Checkpoint preflight has not passed. Model loading was not started.")
ckpts = REPO / "ckpts"
ckpts.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(ckpts)
os.environ["HF_HUB_CACHE"] = str(ckpts)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(ckpts)
print("checkpoint_dir", ckpts)
print("selected_known_bytes", (PREFLIGHT / "checkpoints.ok").read_text(encoding="utf-8").strip())
print("model_load=deferred_to_the_single_generation_process")
(PREFLIGHT / "load_ready.ok").write_text("ok\n", encoding="utf-8")


## 9. Claim one ReelForge job and generate it

The cells above are the verified setup, including `wgp.py --dry-run`. That dry-run prompt is not the clip that gets uploaded.

Set `REELFORGE_API_BASE` to the public https origin of the ReelForge API, and store `REELFORGE_WORKER_TOKEN` as a Kaggle secret of the same name. The next cell reads those and stops if either is missing. It does not print the token.

The generation cell then claims one job from `GET /api/worker/kaggle/jobs/next?provider=wan`. A scene that is not exactly 5 seconds is failed with `retryable` false, and `wgp.py` is not started. Otherwise only the prompt is taken from the job. Generation stays 416×240, 81 frames, 20 steps, seed 1, FP16, SDPA, profile 5, on `cuda:0`.

One MP4 is uploaded to `POST /api/worker/kaggle/jobs/{id}/complete`. A probe mismatch is not uploaded. Reference images are ignored.


In [ ]:
import os
from urllib.parse import urlparse
from kaggle_secrets import UserSecretsClient

if not os.environ.get("REELFORGE_WORKER_TOKEN", "").strip():
    os.environ["REELFORGE_WORKER_TOKEN"] = UserSecretsClient().get_secret("REELFORGE_WORKER_TOKEN")
if not os.environ.get("REELFORGE_API_BASE", "").strip():
    os.environ["REELFORGE_API_BASE"] = UserSecretsClient().get_secret("REELFORGE_API_BASE")
os.environ.setdefault("REELFORGE_WORKER_ID", "wan2gp-1")

base = os.environ["REELFORGE_API_BASE"].strip()
parsed = urlparse(base)
if parsed.scheme != "https" or not parsed.netloc or parsed.path not in ("", "/"):
    raise SystemExit("REELFORGE_API_BASE must be an https origin with no path. Nothing was claimed.")
print("worker_api_host", parsed.hostname)
print("worker_id", os.environ["REELFORGE_WORKER_ID"])


In [ ]:
"""One ReelForge scene, generated by the proven Wan2GP command.

The polling notebook pastes this module and calls ``run_kaggle_session``.
Tests import it directly. Nothing here downloads weights, starts Kaggle, or
imports the Diffusers worker.

Proven settings, from the run that produced an 81-frame H.264 file:

    CUDA_VISIBLE_DEVICES=0 python wgp.py --process <settings.json> \\
        --output-dir <out> --gpu cuda:0 --attention sdpa --profile 5 \\
        --fp16 --verbose 2

Model ``t2v_1.3B``, 416x240, 81 frames, 20 steps, seed 1, profile 5.
81 / 16 = 5.0625 seconds. A scene of any other length is failed before
``wgp.py`` starts, and a failed run does not rewrite those settings.
"""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path
from typing import Any, Callable, Mapping

COMMIT = "b8b18f8114e432eea8f3d7e853a51dd91fa99571"
MODEL_TYPE = "t2v_1.3B"
RESOLUTION = "416x240"
WIDTH = 416
HEIGHT = 240
FRAMES = 81
FPS = 16
STEPS = 20
SEED = 1
ATTENTION = "sdpa"
PROFILE = "5"
GPU = "cuda:0"
SCENE_SECONDS = 5
PROVEN_SECONDS = FRAMES / FPS
PROVIDER = "wan"
WORKER_ID = "wan2gp-1"

Generate = Callable[[Path, Path, Path, Path], int]
Probe = Callable[[Path], dict]


def proven_overrides(prompt: str) -> dict:
    """Settings keys the verified notebook wrote over the factory template.

    Width, height, fps, and quality from the job are ignored. The backend
    asks for the export frame (often 1080x1920 at 30 fps) and normalises
    the upload itself.
    """
    return {
        "model_type": MODEL_TYPE,
        "prompt": prompt,
        "resolution": RESOLUTION,
        "video_length": FRAMES,
        "num_inference_steps": STEPS,
        "seed": SEED,
        "repeat_generation": 1,
        "prompt_enhancer": "",
        "spatial_upsampling": "",
        "temporal_upsampling": "",
        "override_profile": int(PROFILE),
        "override_attention": ATTENTION,
        "image_mode": 0,
    }


def generation_argv(python: str, settings_path: str, output_dir: str) -> list[str]:
    return [
        python, "wgp.py",
        "--process", settings_path,
        "--output-dir", output_dir,
        "--gpu", GPU,
        "--attention", ATTENTION,
        "--profile", PROFILE,
        "--fp16",
        "--verbose", "2",
    ]


def generation_env(base: Mapping[str, str], ckpts: Path) -> dict[str, str]:
    env = dict(base)
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["HF_HOME"] = str(ckpts)
    env["HF_HUB_CACHE"] = str(ckpts)
    env["HUGGINGFACE_HUB_CACHE"] = str(ckpts)
    return env


def _whole_seconds(value: Any) -> int | None:
    if isinstance(value, bool):
        return None
    if isinstance(value, int):
        return value
    if isinstance(value, float) and value.is_integer():
        return int(value)
    return None


def decide(job: Mapping[str, Any]) -> dict:
    """Return a generate decision or a non-retryable failure.

    Does not read width, height, fps, quality, or reference_url, and does
    not offer an alternate frame count.
    """
    seconds = _whole_seconds(job.get("seconds"))
    prompt = job.get("prompt")
    if seconds is None:
        reason = "scene duration must be a whole number of seconds"
    elif seconds != SCENE_SECONDS:
        reason = (
            f"scene is {seconds}s; this worker only generates the proven "
            f"{FRAMES}-frame clip ({PROVEN_SECONDS:.4f}s) for a "
            f"{SCENE_SECONDS}-second scene and will not change settings"
        )
    elif not isinstance(prompt, str) or not prompt.strip():
        reason = "the job has no prompt"
    else:
        return {
            "action": "generate",
            "reason": None,
            "retryable": False,
            "settings": proven_overrides(prompt),
        }
    return {"action": "fail", "reason": reason, "retryable": False, "settings": None}


def apply_proven(settings: dict, prompt: str) -> dict:
    """Overwrite only the proven keys. Leave the rest of the template alone."""
    settings.update(proven_overrides(prompt))
    return settings


def validate_probe(info: Mapping[str, Any]) -> str | None:
    codec = info.get("codec")
    width = info.get("width")
    height = info.get("height")
    frames = str(info.get("frames"))
    rate = info.get("rate")
    try:
        duration = float(info.get("duration"))
    except (TypeError, ValueError):
        duration = 0.0
    if codec != "h264" or width != WIDTH or height != HEIGHT or frames != str(FRAMES) or rate != f"{FPS}/1":
        return f"ffprobe did not match H.264, {WIDTH}x{HEIGHT}, {FRAMES} frames, and {FPS} fps."
    if abs(duration - PROVEN_SECONDS) > 0.02:
        return f"Duration {duration} is not {PROVEN_SECONDS} seconds."
    return None


def probe_media(path: Path) -> dict:
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_format", "-show_streams", "-of", "json", str(path)],
        check=False, text=True, capture_output=True,
    )
    if probe.returncode != 0:
        raise RuntimeError(probe.stderr.strip() or "ffprobe failed")
    info = json.loads(probe.stdout)
    videos = [stream for stream in info.get("streams", []) if stream.get("codec_type") == "video"]
    if len(videos) != 1:
        raise RuntimeError("ffprobe did not find one video stream")
    video = videos[0]
    duration = float(info.get("format", {}).get("duration") or video.get("duration") or 0)
    return {
        "codec": video.get("codec_name"),
        "width": int(video.get("width") or 0),
        "height": int(video.get("height") or 0),
        "frames": video.get("nb_frames"),
        "rate": video.get("r_frame_rate"),
        "duration": duration,
    }


def multipart_payload(filename: str, data: bytes) -> tuple[bytes, str]:
    boundary = "----reelforge-worker"
    head = (
        f"--{boundary}\r\n"
        f'Content-Disposition: form-data; name="file"; filename="{filename}"\r\n'
        f"Content-Type: video/mp4\r\n\r\n"
    ).encode()
    body = head + data + f"\r\n--{boundary}--\r\n".encode()
    return body, f"multipart/form-data; boundary={boundary}"


def failure_payload(detail: str, retryable: bool) -> dict:
    return {"detail": detail[:2000], "retryable": retryable}


class ReelForgeClient:
    """The existing worker API. No new routes."""

    def __init__(self, base: str, token: str, worker_id: str = WORKER_ID):
        if not base:
            raise RuntimeError("REELFORGE_API_BASE is not set")
        if not token:
            raise RuntimeError("REELFORGE_WORKER_TOKEN is not set")
        parsed = urllib.parse.urlparse(base.strip())
        if parsed.scheme != "https" or not parsed.netloc or parsed.path not in ("", "/"):
            raise RuntimeError("REELFORGE_API_BASE must be an https origin with no path")
        self.base = base.strip().rstrip("/")
        self.token = token
        self.worker_id = worker_id

    def _request(self, method: str, path: str, body: dict | None = None, timeout: int = 60):
        data = json.dumps(body).encode() if body is not None else None
        req = urllib.request.Request(self.base + path, data=data, method=method)
        req.add_header("Authorization", f"Bearer {self.token}")
        if data:
            req.add_header("Content-Type", "application/json")
        with urllib.request.urlopen(req, timeout=timeout) as res:
            raw = res.read()
            return json.loads(raw) if raw else {}

    def next_job(self) -> dict | None:
        query = urllib.parse.urlencode({
            "worker_id": self.worker_id, "provider": PROVIDER,
        })
        body = self._request("GET", f"/api/worker/kaggle/jobs/next?{query}")
        return body.get("job")

    def complete(self, job_id: str, clip: Path) -> dict:
        payload, content_type = multipart_payload(clip.name, clip.read_bytes())
        req = urllib.request.Request(
            f"{self.base}/api/worker/kaggle/jobs/{job_id}/complete",
            data=payload, method="POST",
        )
        req.add_header("Authorization", f"Bearer {self.token}")
        req.add_header("Content-Type", content_type)
        try:
            with urllib.request.urlopen(req, timeout=600) as res:
                return json.loads(res.read() or b"{}")
        except urllib.error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")[:500]
            raise RuntimeError(f"upload was rejected ({exc.code}): {detail}") from exc

    def fail(self, job_id: str, detail: str, retryable: bool = False) -> dict:
        return self._request(
            "POST", f"/api/worker/kaggle/jobs/{job_id}/fail",
            failure_payload(detail, retryable),
        )


def default_generate(repo: Path, settings_path: Path, output_dir: Path, log_path: Path) -> int:
    env = generation_env(os.environ, repo / "ckpts")
    cmd = generation_argv(sys.executable, str(settings_path), str(output_dir))
    print("$", "CUDA_VISIBLE_DEVICES=0", " ".join(cmd))
    with log_path.open("w", encoding="utf-8") as log:
        completed = subprocess.run(
            cmd, cwd=repo, env=env, check=False, text=True,
            stdout=log, stderr=subprocess.STDOUT,
        )
    tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-40:]
    print("\n".join(tail))
    print("exit_code", completed.returncode)
    return completed.returncode


def _fail(client: ReelForgeClient, job_id: str, detail: str) -> None:
    try:
        client.fail(job_id, detail, retryable=False)
    except Exception as exc:
        raise SystemExit(f"{detail} (failure report also failed: {exc})") from exc
    raise SystemExit(detail)


def _preserved(root: Path) -> dict[Path, tuple[int, int]]:
    kept = [root / "result.mp4", root / "result_5sec.mp4", root / "result_camera_motion.mp4"]
    return {
        path: (path.stat().st_size, path.stat().st_mtime_ns)
        for path in kept if path.is_file()
    }


def _modified_preserved(stamps: Mapping[Path, tuple[int, int]]) -> str | None:
    for path, stamp in stamps.items():
        if (path.stat().st_size, path.stat().st_mtime_ns) != stamp:
            return f"{path.name} was modified."
    return None


def run_kaggle_session(
    *,
    root: Path | None = None,
    environ: Mapping[str, str] | None = None,
    client: Any | None = None,
    generate: Generate | None = None,
    probe: Probe | None = None,
) -> dict:
    """Claim one wan job, generate once, upload once.

    Stops before ``wgp.py`` when the scene is not 5 seconds, the prompt is
    empty, or the proven settings file is missing. A non-zero exit or a
    probe mismatch is reported with retryable false and does not change
    resolution, frames, steps, seed, attention, or profile.
    """
    root = Path(root or "/kaggle/working")
    preflight = root / "wan2gp_preflight"
    repo = root / "Wan2GP"
    if not (preflight / "load_ready.ok").is_file():
        raise SystemExit("Model-load gate has not passed. Nothing was claimed.")
    output_dir = root / "out"
    already = sorted(output_dir.rglob("*.mp4")) if output_dir.is_dir() else []
    if already or (root / "result_wan2gp.mp4").is_file() or (preflight / "generation.ok").is_file():
        raise SystemExit(
            "This session already has a generation output. Nothing was claimed."
        )

    env = os.environ if environ is None else environ
    base = str(env.get("REELFORGE_API_BASE", "")).strip()
    token = str(env.get("REELFORGE_WORKER_TOKEN", "")).strip()
    if not base or not token:
        raise SystemExit(
            "REELFORGE_API_BASE and REELFORGE_WORKER_TOKEN are required. Nothing was claimed."
        )
    worker_id = str(env.get("REELFORGE_WORKER_ID", WORKER_ID))
    api = client if client is not None else ReelForgeClient(base, token, worker_id)

    job = api.next_job()
    if not job:
        raise SystemExit("No pending wan job. Nothing was generated.")
    job_id = str(job["job_id"])
    decision = decide(job)
    if decision["action"] != "generate":
        _fail(api, job_id, decision["reason"])

    settings_path = root / "t2v_1_3B_settings.json"
    if not settings_path.is_file():
        _fail(api, job_id, "settings file is missing; generation was not started")
    settings = json.loads(settings_path.read_text(encoding="utf-8"))
    prompt = decision["settings"]["prompt"]
    apply_proven(settings, prompt)
    settings_path.write_text(json.dumps(settings, indent=2) + "\n", encoding="utf-8")

    output_dir.mkdir(exist_ok=True)
    kept = _preserved(root)
    log_path = root / "wan2gp_generate.log"
    runner = generate if generate is not None else default_generate
    code = runner(repo, settings_path, output_dir, log_path)
    modified = _modified_preserved(kept)
    if modified:
        _fail(api, job_id, modified)
    if code != 0:
        _fail(api, job_id, f"wgp.py exited {code}. Settings were not changed.")

    mp4s = sorted(output_dir.rglob("*.mp4"))
    if len(mp4s) != 1:
        _fail(api, job_id, f"expected one MP4, found {len(mp4s)}. The file was not uploaded.")
    source = mp4s[0]
    if source.stat().st_size <= 0:
        _fail(api, job_id, "the MP4 is empty. The file was not uploaded.")

    reader = probe if probe is not None else probe_media
    try:
        info = reader(source)
        problem = validate_probe(info)
    except Exception as exc:
        _fail(api, job_id, f"ffprobe failed: {exc}. The file was not uploaded.")
    if problem:
        _fail(api, job_id, f"{problem} The file was not uploaded.")

    destination = root / "result_wan2gp.mp4"
    shutil.copy2(source, destination)
    modified = _modified_preserved(kept)
    if modified:
        _fail(api, job_id, modified)
    if destination.stat().st_size <= 0:
        _fail(api, job_id, "result_wan2gp.mp4 is empty. The file was not uploaded.")
    try:
        api.complete(job_id, destination)
    except Exception as exc:
        _fail(api, job_id, f"upload failed: {exc}. result_wan2gp.mp4 was kept.")

    (preflight / "generation.ok").write_text("ok\n", encoding="utf-8")
    (preflight / "ffprobe.ok").write_text(str(destination) + "\n", encoding="utf-8")
    print("result_mp4", destination)
    print("ffprobe_validation=passed")
    return {"job_id": job_id, "output": str(destination), "prompt": prompt}

run_kaggle_session()


## 10. Session summary

A missing FFprobe sentinel means this run has not verified an MP4. If a cell stops, do not change generation settings and do not start a second generation in the same session.


In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working")
PREFLIGHT = ROOT / "wan2gp_preflight"
print("commit b8b18f8114e432eea8f3d7e853a51dd91fa99571")
print("model t2v_1.3B")
print("resolution 416x240")
print("frames 81")
print("fps 16")
print("steps 20")
print("seed 1")
print("precision fp16")
print("attention sdpa")
print("profile 5")
print("gpu cuda:0")
marker = PREFLIGHT / "ffprobe.ok"
print("ffprobe", marker.read_text(encoding="utf-8").strip() if marker.is_file() else "not verified")
result = ROOT / "result_wan2gp.mp4"
print("result_exists", result.is_file())
print("result_bytes", result.stat().st_size if result.is_file() else 0)
